# Stage 2 zero-transport reconstruction

Select an A100 80 GB runtime and choose **Runtime → Run all**. Authorize the Google Drive mount when prompted. The operator checks the three specified retrospective validation scans, loads the frozen Run C VAE and August bank statistics, and writes one JSON per scan followed by a summary.

The primary value is raw canonical foreground RMSE for the mean-filled arm minus the native arm. Positive values mean mean-filling worsened reconstruction. The third arm restores native values outside the latent valid core.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import tempfile

import torch
import nibabel  # Check the original-NIfTI loader before copying the bank archive.

IMPLEMENTATION = "37063b561d9e9dee4469c23567e5c05afbc5bfa5"
REPOSITORY = "https://github.com/GuillermoTafoya/MRIxFields.git"
if not torch.cuda.is_available():
    raise RuntimeError("Select an A100 80 GB GPU runtime in Colab.")
gpu = torch.cuda.get_device_properties(0)
if "A100" not in gpu.name or gpu.total_memory < 79 * 1024**3:
    raise RuntimeError(f"Expected A100 80 GB, found {gpu.name} ({gpu.total_memory / 1024**3:.1f} GiB).")

repo = Path(tempfile.mkdtemp(prefix="fieldbridge-zero-", dir="/content"))
subprocess.run(["git", "init", "--quiet", str(repo)], check=True)
def git(*args):
    return subprocess.check_output(["git", *args], cwd=repo, text=True).strip()

git("remote", "add", "origin", REPOSITORY)
git("fetch", "--quiet", "--depth=1", "origin", IMPLEMENTATION)
git("checkout", "--quiet", "--detach", "FETCH_HEAD")
if git("rev-parse", "HEAD") != IMPLEMENTATION or git("status", "--porcelain"):
    raise RuntimeError("Pinned implementation checkout did not verify.")
print(f"{gpu.name} | implementation {IMPLEMENTATION}")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
env = dict(os.environ, PYTHONPATH=str(repo / "src"),
           FIELDBRIDGE_IMPLEMENTATION_COMMIT=IMPLEMENTATION,
           PYTHONDONTWRITEBYTECODE="1", PYTHONUNBUFFERED="1")
operator = repo / "notebooks" / "stage2_zero_transport_operator.py"
subprocess.run([sys.executable, "-B", str(operator)], cwd=repo, env=env, check=True)